In [1]:
import os

os.chdir(
    "/Users/adarsh/Desktop/ML_Project"
)


import pandas as pd
import numpy as np


from sklearn.model_selection import train_test_split


from catboost import CatBoostClassifier


from sklearn.utils.class_weight import compute_class_weight


from sklearn.metrics import (
    accuracy_score,
    classification_report
)


import joblib

In [2]:
df = pd.read_csv(
    "datasets/fertilizer_recommendation_preprocessed.csv"
)


df.head()

,soil_ph,soil_moisture,organic_carbon,electrical_conductivity,nitrogen_level,phosphorus_level,potassium_level,temperature,humidity,rainfall,...,region_north,region_south,region_west,recommended_fertilizer_compost,recommended_fertilizer_dap,recommended_fertilizer_mop,recommended_fertilizer_npk,recommended_fertilizer_ssp,recommended_fertilizer_urea,recommended_fertilizer_zinc sulphate
0,6.07,34.98,0.32,1.87,61,44,84,19.84,83.31,1693.22,...,False,True,False,False,False,True,False,False,False,False
1,6.39,47.34,0.28,0.21,59,56,18,24.40,46.27,1030.21,...,False,False,False,False,False,False,False,False,True,False
2,7.92,38.13,0.99,1.88,43,21,119,24.82,71.86,1166.16,...,False,True,False,False,False,False,False,False,True,False
3,5.86,14.17,1.46,0.36,88,46,34,27.87,53.23,2881.83,...,False,False,True,False,False,True,False,False,False,False
4,7.98,19.28,0.85,2.16,104,53,98,24.17,51.87,714.84,...,False,False,False,False,False,False,False,False,False,True


In [3]:
fertilizer_columns = [
    "recommended_fertilizer_compost",
    "recommended_fertilizer_dap",
    "recommended_fertilizer_mop",
    "recommended_fertilizer_npk",
    "recommended_fertilizer_ssp",
    "recommended_fertilizer_urea",
    "recommended_fertilizer_zinc sulphate"
]


y = df[
    fertilizer_columns
].idxmax(axis=1)


y = y.str.replace(
    "recommended_fertilizer_",
    ""
)


X = df.drop(
    fertilizer_columns + ["fertilizer_code"],
    axis=1
)


print(X.shape)
print(y.shape)

(10000, 58)
(10000,)


In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [5]:
classes = np.unique(
    y_train
)


weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train
)


class_weights = dict(
    zip(
        classes,
        weights
    )
)


class_weights

{'compost': np.float64(1.4339487363326762),
 'dap': np.float64(0.4892367906066536),
 'mop': np.float64(1.0149708195889369),
 'npk': np.float64(2.227791701475912),
 'ssp': np.float64(7.8817733990147785),
 'urea': np.float64(0.46064374964012206),
 'zinc sulphate': np.float64(1.8984337921214998)}

In [6]:
fertilizer_cat = CatBoostClassifier(

    iterations=700,

    learning_rate=0.05,

    depth=8,

    l2_leaf_reg=5,

    loss_function="MultiClass",

    class_weights=class_weights,

    random_seed=42,

    verbose=100

)


fertilizer_cat.fit(
    X_train,
    y_train
)

0:	learn: 1.7834772	total: 115ms	remaining: 1m 20s
100:	learn: 0.4383952	total: 2.94s	remaining: 17.4s
200:	learn: 0.3081746	total: 5.53s	remaining: 13.7s
300:	learn: 0.2297231	total: 8.69s	remaining: 11.5s
400:	learn: 0.1794911	total: 11.5s	remaining: 8.55s
500:	learn: 0.1451028	total: 14.9s	remaining: 5.92s
600:	learn: 0.1209649	total: 17.8s	remaining: 2.93s
699:	learn: 0.1030935	total: 20.7s	remaining: 0us


CatBoostClassifier(class_weights={'compost': np.float64(1.4339487363326762), 'dap': np.float64(0.4892367906066536), 'mop': np.float64(1.0149708195889369), 'npk': np.float64(2.227791701475912), 'ssp': np.float64(7.8817733990147785), 'urea': np.float64(0.46064374964012206), 'zinc sulphate': np.float64(1.8984337921214998)}, depth=8, iterations=700, l2_leaf_reg=5, learning_rate=0.05, loss_function='MultiClass', random_seed=42, verbose=100)

In [7]:
prediction = fertilizer_cat.predict(
    X_test
)


accuracy = accuracy_score(
    y_test,
    prediction
)


print(
    "Tuned CatBoost Accuracy:",
    accuracy * 100
)


print(
    classification_report(
        y_test,
        prediction
    )
)

Tuned CatBoost Accuracy: 87.05000000000001
               precision    recall  f1-score   support

      compost       0.85      0.81      0.83       199
          dap       0.99      0.92      0.95       584
          mop       0.92      0.86      0.89       282
          npk       0.64      0.69      0.66       128
          ssp       0.12      0.32      0.17        37
         urea       1.00      0.93      0.96       620
zinc sulphate       0.66      0.81      0.72       150

     accuracy                           0.87      2000
    macro avg       0.74      0.76      0.74      2000
 weighted avg       0.91      0.87      0.89      2000



In [8]:
joblib.dump(
    fertilizer_cat,
    "models/fertilizer_catboost_tuned.pkl"
)

['models/fertilizer_catboost_tuned.pkl']